# Week 07 · Resampling Foundations: Bootstrap and Permutation Tests

*DS 207 · Week 07*

**Topic:** Resampling foundations: bootstrap standard errors and confidence intervals, permutation tests, and choosing between analytic and resampling methods

**Business scenario:** Cebu Bus Rapid Transit Operations (the Authority) is reviewing service on Corridor 2 during the weekday evening peak. Carla Ouano, Service Planning Manager, relies on passenger wait-to-board times (minutes from tap-in at the stop to boarding), which are right-skewed with occasional very long waits. Her decisions concern how precisely the Authority can report its wait-time figures, whether a new boarding arrangement helps, and whether a revised dispatch policy justifies its cost.

**Author:** EC Corro, built in collaboration with Claude, ChatGPT, and Gemini

**Submission instructions:** Save this notebook as `ds207_07_resampling_<lastname>.ipynb` and submit it to your shared Google Drive folder by 11:59 PM on the day after the associated session, unless an alternative deadline is announced.

**AI use policy:** Generative AI tools may be used for concept clarification, pseudocode, debugging, and idea generation. You must be able to explain any submitted answer without the tool, and you must cite any AI use.

**Points:** Section 1 (30 pts) + Section 2 (30 pts) = 60 pts.

## Learning Objectives

This assignment assesses whether you can:

- Compute and interpret bootstrap standard errors, by hand on a small extract and in Python at full scale.
- Construct percentile bootstrap intervals and recognise when they are unreliable.
- Carry out a permutation test for a two-group comparison and report p-value and effect size together.
- Translate resampling results into a recommendation that a transit planner could act on.

## Section 1: Concepts, Computation & Coding

**30 points: three items worth 10 points each.** Within every item the split is (a) 3 points, (b) 4 points, (c) 3 points. Show your work for every by-hand computation; a correct number without supporting steps earns partial credit only. Run the first code cell before starting; it loads the data used throughout both sections.

In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook")

# Corridor 2, weekday evening peak: wait-to-board time in minutes
eve_waits = np.array([4.1, 5.3, 3.1, 3.5, 8.2, 3.4, 6.1, 17.4, 4.1, 59.5, 2.6, 28.6, 10.4, 3.9, 15.3, 11.2, 3.6, 3.9, 8.6, 13.0, 4.3, 1.8, 6.4, 14.6, 4.9, 21.7, 1.9, 14.1, 4.4, 8.3, 15.8, 11.1, 10.9, 4.1, 8.6, 4.7, 2.7, 2.0, 2.8, 11.3, 6.3, 7.2, 5.4, 6.4, 7.0])            # 45 passengers, all stops

lane_waits = np.array([4.9, 10.2, 13.1, 13.0, 20.8, 5.6, 9.7, 5.6, 3.0, 4.6, 7.6, 4.0, 6.2, 13.7, 9.9, 9.9, 8.6, 6.3, 2.3, 12.7, 2.8, 8.0, 6.6, 7.6, 8.1, 5.7, 12.0, 3.2, 11.3, 2.5, 14.3, 5.2, 8.8, 17.3, 1.9, 5.8, 9.9, 12.0])   # 38 passengers at stops with pre-boarding queue lanes

nolane_waits = np.array([26.3, 15.0, 14.0, 10.3, 11.6, 10.0, 20.0, 10.7, 15.7, 31.9, 22.8, 7.3, 8.8, 16.5, 5.9, 2.4, 4.4, 14.2, 8.1, 7.3, 16.6, 4.3, 51.9, 9.7, 7.4, 43.8, 4.2, 6.5, 8.9, 4.6, 7.4, 14.7, 10.8, 9.2, 19.9, 39.1, 30.7, 2.0, 26.6, 17.0, 12.9, 4.5])   # 42 passengers at stops without queue lanes

# New Feeder Stop F, evening peak week: 16 passengers
feeder_waits = np.array([7.2, 46.0, 4.2, 2.6, 6.4, 8.0, 4.9, 5.9, 7.1, 18.4, 7.8, 15.1, 8.4, 19.4, 5.9, 12.8])

# Revised-headway pilot: average passenger wait per trip (minutes), 10 trips under each policy
policy_current = np.array([10.4, 13.3, 14.3, 14.3, 16.2, 10.9, 13.1, 10.9, 8.4, 10.1])
policy_revised = np.array([10.3, 7.8, 9.5, 12.7, 11.4, 11.4, 10.8, 9.6, 5.5, 12.4])

### Item 1: Bootstrap Standard Errors (10 points)

Carla wants a margin of error for the median wait-to-board time on Corridor 2 in the evening peak. The array `eve_waits` holds the waits of 45 sampled passengers.

**(a) Concept (3 points).** Explain why the bootstrap is an appropriate way to obtain the standard error of the median here, whereas the standard error of the mean can be obtained from a formula. In your answer, state what resampling with replacement is meant to mimic and what the bootstrap distribution is, and is not, centred on.

**Answer (a)**

**(b) By hand (4 points).** Consider an extract of three passengers with waits $\{3, 8, 19\}$ minutes. Show all steps.

1. (2 points) Compute the exact bootstrap standard error of the sample mean using the plug-in variance $\hat\sigma^2/n$ with divisor $n$, and compare it with the analytic $s/\sqrt{n}$.
2. (2 points) List the exact bootstrap distribution of the sample median over the $3^3 = 27$ equally likely resamples (each possible value with its probability), and compute its standard error.

**Answer (b)**

**(c) Code (3 points).** Verify the idea at full scale on `eve_waits`.

1. (1 point) With $B = 10{,}000$ resamples and a fixed random seed, compute the bootstrap standard errors of the mean and of the median.
2. (1 point) Compare the bootstrap standard error of the mean with the analytic $s/\sqrt{n}$.
3. (1 point) In two or three sentences, state which of the two statistics is less affected by the very long waits in this sample, using your standard errors as evidence.

In [ ]:
# TODO

**Answer (c) interpretation**

### Item 2: Percentile Intervals and Where They Fail (10 points)

A colleague proposes that the Authority publish a 95% percentile bootstrap interval for the **90th percentile** of the wait-to-board time, using a 12-passenger pilot sample.

**(a) Concept (3 points).** Give two distinct reasons why a percentile bootstrap interval for a 90th percentile is likely to be unreliable with only 12 observations. Refer to what a bootstrap resample can and cannot contain.

**Answer (a)**

**(b) By hand (4 points).** A small sample of five passengers has waits $\{4, 6, 9, 15, 38\}$ minutes, and its median is 9. Twenty bootstrap resamples were drawn and the median of each was recorded:

$$
6,\ 9,\ 15,\ 6,\ 6,\ 6,\ 9,\ 9,\ 38,\ 6,\ 6,\ 6,\ 9,\ 38,\ 6,\ 6,\ 15,\ 9,\ 38,\ 6 .
$$

1. (2 points) Sort the 20 values and give the approximate 90% percentile interval for the median by discarding the lowest and the highest value, that is, reading off the 2nd smallest and the 19th smallest. Comment on what the interval tells you.
2. (2 points) Compute the probability that a bootstrap resample of this sample has a maximum equal to the observed maximum of 38, using $1 - (1 - 1/n)^n$, and explain what this implies for statistics that depend on the upper tail.

**Answer (b)**

**(c) Code (3 points).** Using `eve_waits`:

1. (1 point) Compute a 95% percentile bootstrap interval ($B = 10{,}000$, fixed seed) for the median wait using all 45 observations.
2. (1 point) Compute 95% percentile bootstrap intervals for the 90th percentile using (i) all 45 observations and (ii) only the first 12, `eve_waits[:12]`. Report the width of each interval and the number of distinct values taken by the bootstrap 90th percentile in the 12-observation case.
3. (1 point) In two or three sentences, interpret the comparison and say whether you would publish the 12-observation interval.

In [ ]:
# TODO

**Answer (c) interpretation**

### Item 3: Permutation Test for a Two-Group Difference (10 points)

The Authority introduced pre-boarding queue lanes at some Corridor 2 stops. Carla asks whether passengers at stops **with** queue lanes board sooner than passengers at stops **without**. The arrays `lane_waits` (38 passengers) and `nolane_waits` (42 passengers) hold wait-to-board times in minutes from the same evening-peak window. The Authority reports medians, and two waits in the no-lane group exceed 40 minutes.

**(a) Concept (3 points).** State the null hypothesis of a permutation test for this comparison, explain what it means for the group labels to be exchangeable under that null, and give one reason a permutation test is preferable to a two-sample $t$-test here. Name one feature of how the lanes were assigned that could undermine exchangeability.

**Answer (a)**

**(b) By hand (4 points).** Take an extract of five passengers: no-lane waits $\{9, 14\}$ and lane waits $\{3, 5, 8\}$. Use the difference in means, $T = \bar x_{\text{no lane}} - \bar x_{\text{lane}}$.

1. (1 point) Compute the observed $T$.
2. (2 points) Enumerate all $\binom{5}{2} = 10$ ways of assigning two of the five values to the "no lane" label and compute $T^*$ for each.
3. (1 point) Report the one-sided p-value (share of relabelings with $T^* \ge T_{\text{obs}}$) and the two-sided p-value (share with $|T^*| \ge |T_{\text{obs}}|$), and state the smallest p-value this design could ever produce.

**Answer (b)**

**(c) Code (3 points).** Using the full arrays:

1. (1 point) Run a permutation test on the difference in medians, $T = \tilde x_{\text{no lane}} - \tilde x_{\text{lane}}$, with at least 10,000 random relabelings and a fixed seed. Report the observed difference and the two-sided p-value, and plot the null distribution with the observed value marked.
2. (1 point) Report the Welch $t$-test p-value and the Mann-Whitney p-value for comparison, and a bootstrap 95% percentile interval for the difference in medians (resample each group separately).
3. (1 point) In three sentences or fewer, interpret the permutation p-value and the interval together.

In [ ]:
# TODO

**Answer (c) interpretation**

## Section 2: Business Context & Reasoning

**30 points: two items worth 15 points each.** Within each item the split is:

- **Assumption (3 points):** a stated assumption, with a reason it is plausible or a way it could fail in this setting.
- **Technique and interpretation (6 points):** an appropriate resampling method, correctly applied, with the result stated and interpreted, including how far the method can be trusted for this sample.
- **Recommendation (6 points):** a concrete recommendation to the stakeholder that follows from your result and weighs the consequences of being wrong.

Technical soundness accounts for 9 of the 15 points and business reasoning for 6. A recommendation that does not follow from the result loses marks even when the calculation is correct, and a sensible-sounding recommendation resting on an incorrect or unsupported calculation loses marks as well. Write your responses in the cells provided; use code cells for computation and markdown cells for explanation.

### Item 4: A Published Service Standard for Feeder Stop F (15 points)

The Authority wants to publish a service standard of the form "nine in ten passengers board within $X$ minutes" for the new Feeder Stop F during the evening peak. Three candidate values are under discussion: $X = 12$, $X = 15$ and $X = 20$ minutes. A published standard that is missed triggers a public report to the city council, while a standard set too loosely is criticised as meaningless. The only data available are the wait-to-board times of 16 passengers observed during one evening-peak week, stored in `feeder_waits`.

Carla asks: **which standard, if any, should we publish now, and what should we do next?**

**Assumption (3 points).** State one assumption you are making about the data or the method and why it is reasonable or how it could fail.

**Assumption**

**Technique and interpretation (6 points).** Apply an appropriate resampling method to the data, report the result, and interpret it, including how far the method can be trusted for a sample of this size.

In [ ]:
# TODO

**Interpretation**

**Recommendation (6 points).** Advise Carla on which standard to publish, if any, and on what to do next, justified by your result and by the consequences of being wrong.

**Recommendation**

### Item 5: Rollout of a Revised Headway Policy on Corridor 2 (15 points)

Corridor 2 piloted a revised headway policy on 10 trips and kept the current policy on 10 comparable trips during the same evening peak. The response for each trip is the average passenger wait (minutes); the data are in `policy_current` and `policy_revised`. Rolling the revised policy out across the corridor would cost PHP 1.8 million one-time for software configuration and driver retraining, and the planning team judges the investment worthwhile only if it lowers the average trip wait by **at least 1.5 minutes**.

Carla asks: **should we roll out the revised policy on the basis of this pilot?**

**Assumption (3 points).** State one assumption you are making about the data or the method and why it is reasonable or how it could fail.

**Assumption**

**Technique and interpretation (6 points).** Apply appropriate resampling methods to the data, report the results, and interpret them in relation to the 1.5-minute threshold, including how far the methods can be trusted for a pilot of this size.

In [ ]:
# TODO

**Interpretation**

**Recommendation (6 points).** Advise Carla on whether to roll out, extend the pilot, stage the rollout, or decline, justified by your results and by the costs of a wrong decision in each direction.

**Recommendation**

**Citation Reminder.** All external sources, including textbooks, papers, articles, code snippets and any use of AI tools, must be cited according to course policy. Cite them where they are used and list them at the end of your submission.

## References

Davison, A. C., & Hinkley, D. V. (1997). *Bootstrap methods and their application*. Cambridge University Press.

Efron, B., & Tibshirani, R. J. (1993). *An introduction to the bootstrap*. Chapman & Hall/CRC.

Ernst, M. D. (2004). Permutation methods: A basis for exact inference. *Statistical Science, 19*(4), 676-685.

Good, P. I. (2005). *Permutation, parametric, and bootstrap tests of hypotheses* (3rd ed.). Springer.

Hesterberg, T. C. (2015). What teachers should know about the bootstrap: Resampling in the undergraduate statistics curriculum. *The American Statistician, 69*(4), 371-386.